In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
spark = (
    SparkSession.builder
    .appName("Gold Layer")
    .getOrCreate()
)

In [ ]:
from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=True
)

Mounted at /content/drive


In [ ]:
BASE_PATH = "/content/drive/MyDrive/TechChallenge_Fase2"

SILVER_PATH = f"{BASE_PATH}/data/silver_bigquery"
GOLD_PATH = f"{BASE_PATH}/data/gold"

In [ ]:
municipio = spark.read.parquet(
    f"{SILVER_PATH}/municipio"
)

uf = spark.read.parquet(
    f"{SILVER_PATH}/uf"
)

meta_municipio = spark.read.parquet(
    f"{SILVER_PATH}/meta_municipio"
)

meta_uf = spark.read.parquet(
    f"{SILVER_PATH}/meta_uf"
)

meta_brasil = spark.read.parquet(
    f"{SILVER_PATH}/meta_brasil"
)

alunos_agregados = spark.read.parquet(
    f"{SILVER_PATH}/alunos_agregados"
)

In [ ]:
tabelas_silver = {
    "municipio": municipio,
    "uf": uf,
    "meta_municipio": meta_municipio,
    "meta_uf": meta_uf,
    "meta_brasil": meta_brasil,
    "alunos_agregados": alunos_agregados
}

for nome, df in tabelas_silver.items():
    print(
        f"{nome}: {df.count():,} registros "
        f"e {len(df.columns)} colunas"
    )

municipio: 23,995 registros e 20 colunas
uf: 145 registros e 20 colunas
meta_municipio: 10,704 registros e 17 colunas
meta_uf: 81 registros e 16 colunas
meta_brasil: 3 registros e 15 colunas
alunos_agregados: 79,273 registros e 18 colunas


In [ ]:
print("Municípios:", municipio.count())
print("UF:", uf.count())
print("Meta Município:", meta_municipio.count())
print("Meta UF:", meta_uf.count())
print("Meta Brasil:", meta_brasil.count())

Municípios: 23995
UF: 145
Meta Município: 10704
Meta UF: 81
Meta Brasil: 3


In [ ]:
gold_municipio = municipio

In [ ]:
gold_municipio = (

    gold_municipio

    .withColumn(

        "indice_geral",

        round(

            (
                col("taxa_alfabetizacao") +
                (col("media_portugues") / 10)
            ) / 2,

            2

        )

    )

)

In [ ]:
gold_municipio = (

    gold_municipio

    .withColumn(

        "faixa_portugues",

        when(col("media_portugues") >= 800, "Muito Alta")
        .when(col("media_portugues") >= 700, "Alta")
        .when(col("media_portugues") >= 600, "Média")
        .otherwise("Baixa")

    )

)

In [ ]:
gold_municipio = (

    gold_municipio

    .withColumn(

        "categoria_rede",

        when(col("rede") == 0, "Total")
        .when(col("rede") == 2, "Estadual")
        .when(col("rede") == 3, "Municipal")
        .when(col("rede") == 5, "Privada")
        .otherwise("Não Informado")

    )

)

In [ ]:
gold_municipio = (

    gold_municipio

    .withColumn(

        "ano_referencia",

        col("ano")

    )

)

In [ ]:
gold_municipio.select(

    "ano",
    "id_municipio",
    "categoria_rede",
    "taxa_alfabetizacao",
    "media_portugues",
    "indice_geral",
    "nivel_desempenho",
    "faixa_portugues"

).show(20, truncate=False)

{"ts": "2026-07-14 05:08:55.271", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `nivel_desempenho` cannot be resolved. Did you mean one of the following? [`ano_referencia`, `indice_geral`, `rede`, `serie`, `data_ingestao`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor58.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o181.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `nivel_desempenho` cannot be resolved. Did you mean one of the following? [`ano_referencia`, `indice_geral`, `rede`, `serie`, `data_ingestao`]. SQLSTATE: 42703;\n'Project [ano#19, id_municipio#0, categoria_rede#347, taxa_alfabetizacao#3, media_portu

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `nivel_desempenho` cannot be resolved. Did you mean one of the following? [`ano_referencia`, `indice_geral`, `rede`, `serie`, `data_ingestao`]. SQLSTATE: 42703;
'Project [ano#19, id_municipio#0, categoria_rede#347, taxa_alfabetizacao#3, media_portugues#4, indice_geral#345, 'nivel_desempenho, faixa_portugues#346]
+- Project [id_municipio#0, serie#1, rede#2, taxa_alfabetizacao#3, media_portugues#4, proporcao_aluno_nivel_0#5, proporcao_aluno_nivel_1#6, proporcao_aluno_nivel_2#7, proporcao_aluno_nivel_3#8, proporcao_aluno_nivel_4#9, proporcao_aluno_nivel_5#10, proporcao_aluno_nivel_6#11, proporcao_aluno_nivel_7#12, proporcao_aluno_nivel_8#13, data_ingestao#14, origem#15, modo_ingestao#16, projeto_execucao#17, categoria_rede#347, ano#19, indice_geral#345, faixa_portugues#346, ano#19 AS ano_referencia#348]
   +- Project [id_municipio#0, serie#1, rede#2, taxa_alfabetizacao#3, media_portu

In [ ]:
gold_municipio.printSchema()

root
 |-- id_municipio: string (nullable = true)
 |-- serie: integer (nullable = true)
 |-- rede: integer (nullable = true)
 |-- taxa_alfabetizacao: double (nullable = true)
 |-- media_portugues: double (nullable = true)
 |-- proporcao_aluno_nivel_0: double (nullable = true)
 |-- proporcao_aluno_nivel_1: double (nullable = true)
 |-- proporcao_aluno_nivel_2: double (nullable = true)
 |-- proporcao_aluno_nivel_3: double (nullable = true)
 |-- proporcao_aluno_nivel_4: double (nullable = true)
 |-- proporcao_aluno_nivel_5: double (nullable = true)
 |-- proporcao_aluno_nivel_6: double (nullable = true)
 |-- proporcao_aluno_nivel_7: double (nullable = true)
 |-- proporcao_aluno_nivel_8: double (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- origem: string (nullable = true)
 |-- modo_ingestao: string (nullable = true)
 |-- projeto_execucao: string (nullable = true)
 |-- categoria_rede: string (nullable = false)
 |-- ano: integer (nullable = true)
 |-- indice_geral: dou

In [ ]:
gold_municipio.select(

    round(avg("taxa_alfabetizacao"),2).alias("Média Alfabetização"),

    round(avg("media_portugues"),2).alias("Média Português"),

    round(avg("indice_geral"),2).alias("Índice Geral Médio"),

    count("*").alias("Total de Registros")

).show()

+-------------------+---------------+------------------+------------------+
|Média Alfabetização|Média Português|Índice Geral Médio|Total de Registros|
+-------------------+---------------+------------------+------------------+
|              61.44|         751.38|             68.29|             23995|
+-------------------+---------------+------------------+------------------+



In [ ]:
from pyspark.sql.functions import col, round, when

gold_municipio = (
    municipio
    .withColumn(
        "indice_geral",
        round(
            (
                col("taxa_alfabetizacao")
                + (col("media_portugues") / 10)
            ) / 2,
            2
        )
    )
    .withColumn(
        "nivel_desempenho",
        when(
            col("taxa_alfabetizacao").isNull(),
            "Não informado"
        )
        .when(
            col("taxa_alfabetizacao") >= 80,
            "Excelente"
        )
        .when(
            col("taxa_alfabetizacao") >= 70,
            "Bom"
        )
        .when(
            col("taxa_alfabetizacao") >= 60,
            "Regular"
        )
        .otherwise(
            "Crítico"
        )
    )
    .withColumn(
        "faixa_portugues",
        when(
            col("media_portugues").isNull(),
            "Não informado"
        )
        .when(
            col("media_portugues") >= 800,
            "Muito alta"
        )
        .when(
            col("media_portugues") >= 700,
            "Alta"
        )
        .when(
            col("media_portugues") >= 600,
            "Média"
        )
        .otherwise(
            "Baixa"
        )
    )
    .withColumn(
        "ano_referencia",
        col("ano")
    )
)

In [ ]:
from pyspark.sql.functions import col, when

gold_municipio = (
    gold_municipio
    .withColumn(
        "nivel_desempenho",
        when(
            col("taxa_alfabetizacao").isNull(),
            "Não informado"
        )
        .when(
            col("taxa_alfabetizacao") >= 80,
            "Excelente"
        )
        .when(
            col("taxa_alfabetizacao") >= 70,
            "Bom"
        )
        .when(
            col("taxa_alfabetizacao") >= 60,
            "Regular"
        )
        .otherwise(
            "Crítico"
        )
    )
)

In [ ]:
gold_municipio.groupBy(

    "nivel_desempenho"

).count().orderBy("nivel_desempenho").show()

+----------------+-----+
|nivel_desempenho|count|
+----------------+-----+
|             Bom| 4065|
|         Crítico|10987|
|       Excelente| 4636|
|         Regular| 4307|
+----------------+-----+



In [ ]:
gold_municipio.groupBy(

    "categoria_rede"

).count().orderBy("categoria_rede").show()

+--------------+-----+
|categoria_rede|count|
+--------------+-----+
|      Estadual| 2235|
|     Municipal|10896|
|       Privada|10466|
|         Total|  398|
+--------------+-----+



In [ ]:
gold_municipio.select(

    [
        count(when(col(c).isNull(), c)).alias(c)
        for c in gold_municipio.columns
    ]

).show(truncate=False)

+------------+-----+----+------------------+---------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-------------+------+-------------+----------------+--------------+---+------------+----------------+---------------+--------------+
|id_municipio|serie|rede|taxa_alfabetizacao|media_portugues|proporcao_aluno_nivel_0|proporcao_aluno_nivel_1|proporcao_aluno_nivel_2|proporcao_aluno_nivel_3|proporcao_aluno_nivel_4|proporcao_aluno_nivel_5|proporcao_aluno_nivel_6|proporcao_aluno_nivel_7|proporcao_aluno_nivel_8|data_ingestao|origem|modo_ingestao|projeto_execucao|categoria_rede|ano|indice_geral|nivel_desempenho|faixa_portugues|ano_referencia|
+------------+-----+----+------------------+---------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+--

In [ ]:
gold_municipio.write.mode("overwrite").parquet(

    f"{GOLD_PATH}/desempenho_municipio"

)

print(" Data Mart desempenho_municipio salvo com sucesso!")

 Data Mart desempenho_municipio salvo com sucesso!


In [ ]:
gold_municipio.select("media_portugues").describe().show()

+-------+------------------+
|summary|   media_portugues|
+-------+------------------+
|  count|             23995|
|   mean| 751.3757078891416|
| stddev|23.232836703593563|
|    min|          673.2983|
|    max|          868.4554|
+-------+------------------+



In [ ]:
gold_municipio.select(
    min("media_portugues").alias("mínimo"),
    max("media_portugues").alias("máximo"),
    avg("media_portugues").alias("média")
).show()

+--------+--------+-----------------+
|  mínimo|  máximo|            média|
+--------+--------+-----------------+
|673.2983|868.4554|751.3757078891416|
+--------+--------+-----------------+



In [ ]:
gold_municipio = (

    gold_municipio

    .withColumn(

        "ano_letivo",

        col("ano")

    )

)

In [ ]:
gold_municipio = (

    gold_municipio

    .withColumn(

        "categoria_indice",

        when(col("indice_geral") >= 80, "Muito Alto")
        .when(col("indice_geral") >= 70, "Alto")
        .when(col("indice_geral") >= 60, "Médio")
        .otherwise("Baixo")

    )

)

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

janela = Window.orderBy(col("indice_geral").desc())

gold_municipio = (

    gold_municipio

    .withColumn(

        "ranking_municipio",

        dense_rank().over(janela)

    )

)

In [ ]:
gold_uf = uf

In [ ]:
from pyspark.sql.functions import when, col

gold_uf = (

    gold_uf

    .withColumn(

        "categoria_rede",

        when(col("rede") == 0, "Total")
        .when(col("rede") == 2, "Estadual")
        .when(col("rede") == 3, "Municipal")
        .when(col("rede") == 5, "Privada")
        .otherwise("Não Informado")

    )

)

In [ ]:
gold_uf = (

    gold_uf

    .withColumn(

        "indice_geral",

        round(

            (
                col("taxa_alfabetizacao") +
                (col("media_portugues") / 10)
            ) / 2,

            2

        )

    )

)

In [ ]:
gold_uf = (

    gold_uf

    .withColumn(

        "nivel_desempenho",

        when(col("taxa_alfabetizacao") >= 80, "Excelente")
        .when(col("taxa_alfabetizacao") >= 70, "Bom")
        .when(col("taxa_alfabetizacao") >= 60, "Regular")
        .otherwise("Crítico")

    )

)

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank

janela_uf = Window.partitionBy("ano").orderBy(col("indice_geral").desc())

gold_uf = (

    gold_uf

    .withColumn(

        "ranking_uf",

        dense_rank().over(janela_uf)

    )

)

In [ ]:
gold_uf = (

    gold_uf

    .withColumn(

        "faixa_ranking",

        when(col("ranking_uf") <= 5, "Top 5")
        .when(col("ranking_uf") <= 10, "Top 10")
        .when(col("ranking_uf") <= 20, "Top 20")
        .otherwise("Demais")

    )

)

In [ ]:
gold_uf.select(

    "ano",
    "sigla_uf",
    "categoria_rede",
    "taxa_alfabetizacao",
    "media_portugues",
    "indice_geral",
    "nivel_desempenho",
    "ranking_uf",
    "faixa_ranking"

).orderBy("ano", "ranking_uf").show(30, truncate=False)

+----+--------+--------------+------------------+---------------+------------+----------------+----------+-------------+
|ano |sigla_uf|categoria_rede|taxa_alfabetizacao|media_portugues|indice_geral|nivel_desempenho|ranking_uf|faixa_ranking|
+----+--------+--------------+------------------+---------------+------------+----------------+----------+-------------+
|2023|CE      |Municipal     |84.49             |795.7293       |82.03       |Excelente       |1         |Top 5        |
|2023|CE      |Privada       |84.46             |795.675        |82.01       |Excelente       |2         |Top 5        |
|2023|PR      |Estadual      |80.29             |770.8324       |78.69       |Excelente       |3         |Top 5        |
|2023|CE      |Estadual      |78.75             |785.1658       |78.63       |Bom             |4         |Top 5        |
|2023|PE      |Estadual      |79.25             |763.6886       |77.81       |Bom             |5         |Top 5        |
|2023|GO      |Estadual      |76

In [ ]:
gold_uf.select(

    round(avg("taxa_alfabetizacao"), 2).alias("Média Alfabetização"),

    round(avg("media_portugues"), 2).alias("Média Português"),

    round(avg("indice_geral"), 2).alias("Índice Geral Médio"),

    count("*").alias("Total de Registros")

).show()

+-------------------+---------------+------------------+------------------+
|Média Alfabetização|Média Português|Índice Geral Médio|Total de Registros|
+-------------------+---------------+------------------+------------------+
|              56.37|         745.34|             65.45|               145|
+-------------------+---------------+------------------+------------------+



In [ ]:
gold_uf.filter(

    col("ranking_uf") <= 10

).select(

    "ano",
    "ranking_uf",
    "sigla_uf",
    "indice_geral",
    "taxa_alfabetizacao"

).orderBy("ano", "ranking_uf").show(30, truncate=False)

+----+----------+--------+------------+------------------+
|ano |ranking_uf|sigla_uf|indice_geral|taxa_alfabetizacao|
+----+----------+--------+------------+------------------+
|2023|1         |CE      |82.03       |84.49             |
|2023|2         |CE      |82.01       |84.46             |
|2023|3         |PR      |78.69       |80.29             |
|2023|4         |CE      |78.63       |78.75             |
|2023|5         |PE      |77.81       |79.25             |
|2023|6         |GO      |76.48       |76.59             |
|2023|7         |ES      |75.45       |74.58             |
|2023|8         |MA      |74.59       |72.38             |
|2023|9         |PR      |74.42       |73.12             |
|2023|10        |PR      |74.41       |73.11             |
|2024|1         |CE      |82.54       |85.35             |
|2024|2         |CE      |82.52       |85.31             |
|2024|3         |ES      |81.76       |86.21             |
|2024|4         |GO      |80.15       |83.66            

In [ ]:
gold_uf.write.mode("overwrite").parquet(

    f"{GOLD_PATH}/desempenho_uf"

)

print(" Data Mart desempenho_uf salvo com sucesso")

 Data Mart desempenho_uf salvo com sucesso


In [ ]:
gold_meta_municipio = meta_municipio

In [ ]:
gold_meta_municipio = (

    gold_meta_municipio

    .withColumn(

        "crescimento_meta",

        round(

            col("meta_alfabetizacao_2030") -
            col("meta_alfabetizacao_2024"),

            2

        )

    )

)

In [ ]:
gold_meta_municipio = (

    gold_meta_municipio

    .withColumn(

        "percentual_evolucao",

        round(

            (
                (
                    col("meta_alfabetizacao_2030") -
                    col("meta_alfabetizacao_2024")
                )

                /

                col("meta_alfabetizacao_2024")

            ) * 100,

            2

        )

    )

)

In [ ]:
gold_meta_municipio = (

    gold_meta_municipio

    .withColumn(

        "nivel_meta",

        when(
            col("meta_alfabetizacao_2030") >= 80,
            "Meta Nacional"
        )

        .when(
            col("meta_alfabetizacao_2030") >= 70,
            "Alta"
        )

        .when(
            col("meta_alfabetizacao_2030") >= 60,
            "Média"
        )

        .otherwise(
            "Baixa"
        )

    )

)

In [ ]:
gold_meta_municipio = (

    gold_meta_municipio

    .withColumn(

        "faixa_participacao",

        when(
            col("percentual_participacao") >= 90,
            "Muito Alta"
        )

        .when(
            col("percentual_participacao") >= 75,
            "Alta"
        )

        .when(
            col("percentual_participacao") >= 50,
            "Média"
        )

        .otherwise(
            "Baixa"
        )

    )

)

In [ ]:
gold_meta_municipio.select(

    "ano",
    "id_municipio",
    "meta_alfabetizacao_2024",
    "meta_alfabetizacao_2030",
    "crescimento_meta",
    "percentual_evolucao",
    "nivel_meta",
    "faixa_participacao"

).show(20, truncate=False)

+----+------------+-----------------------+-----------------------+----------------+-------------------+-------------+------------------+
|ano |id_municipio|meta_alfabetizacao_2024|meta_alfabetizacao_2030|crescimento_meta|percentual_evolucao|nivel_meta   |faixa_participacao|
+----+------------+-----------------------+-----------------------+----------------+-------------------+-------------+------------------+
|2024|2210623     |33.21                  |80.0                   |46.79           |140.89             |Meta Nacional|Muito Alta        |
|2024|2705200     |37.86                  |80.0                   |42.14           |111.3              |Meta Nacional|Muito Alta        |
|2024|3522901     |41.78                  |80.0                   |38.22           |91.48              |Meta Nacional|Alta              |
|2024|2905909     |43.29                  |80.0                   |36.71           |84.8               |Meta Nacional|Alta              |
|2024|3171006     |49.64          

In [ ]:
gold_meta_municipio.select(

    round(avg("crescimento_meta"), 2).alias("Crescimento Médio"),

    round(avg("percentual_evolucao"), 2).alias("% Evolução Médio"),

    count("*").alias("Municípios")

).show()

+-----------------+----------------+----------+
|Crescimento Médio|% Evolução Médio|Municípios|
+-----------------+----------------+----------+
|            17.85|           40.22|     10704|
+-----------------+----------------+----------+



In [ ]:
gold_meta_municipio.groupBy(

    "nivel_meta"

).count().show()

+-------------+-----+
|   nivel_meta|count|
+-------------+-----+
|Meta Nacional|10704|
+-------------+-----+



In [ ]:
gold_meta_municipio.write.mode("overwrite").parquet(

    f"{GOLD_PATH}/meta_municipio"

)

print(" Data Mart meta_municipio salvo com sucesso")

 Data Mart meta_municipio salvo com sucesso


In [ ]:
gold_meta_uf = meta_uf

In [ ]:
gold_meta_uf = (
    gold_meta_uf
    .withColumn(
        "crescimento_meta_2024_2030",
        round(
            col("meta_alfabetizacao_2030")
            - col("meta_alfabetizacao_2024"),
            2
        )
    )
)

In [ ]:
gold_meta_uf = (
    gold_meta_uf
    .withColumn(
        "percentual_evolucao_meta",
        when(
            col("meta_alfabetizacao_2024").isNull()
            | (col("meta_alfabetizacao_2024") == 0),
            None
        ).otherwise(
            round(
                (
                    (
                        col("meta_alfabetizacao_2030")
                        - col("meta_alfabetizacao_2024")
                    )
                    / col("meta_alfabetizacao_2024")
                ) * 100,
                2
            )
        )
    )
)


In [ ]:
gold_meta_uf = (
    gold_meta_uf
    .withColumn(
        "diferenca_meta_2024",
        round(
            col("taxa_alfabetizacao")
            - col("meta_alfabetizacao_2024"),
            2
        )
    )
)

In [ ]:
gold_meta_uf = (
    gold_meta_uf
    .withColumn(
        "percentual_cumprimento_meta_2024",
        when(
            col("meta_alfabetizacao_2024").isNull()
            | (col("meta_alfabetizacao_2024") == 0),
            None
        ).otherwise(
            round(
                (
                    col("taxa_alfabetizacao")
                    / col("meta_alfabetizacao_2024")
                ) * 100,
                2
            )
        )
    )
)

In [ ]:
gold_meta_uf = (
    gold_meta_uf
    .withColumn(
        "status_meta_2024",
        when(
            col("meta_alfabetizacao_2024").isNull(),
            "Sem meta disponível"
        )
        .when(
            col("diferenca_meta_2024") >= 0,
            "Meta atingida"
        )
        .when(
            col("diferenca_meta_2024") >= -2,
            "Próximo da meta"
        )
        .otherwise(
            "Abaixo da meta"
        )
    )
)

In [ ]:
gold_meta_uf = (
    gold_meta_uf
    .withColumn(
        "faixa_participacao",
        when(
            col("percentual_participacao").isNull(),
            "Não informado"
        )
        .when(
            col("percentual_participacao") >= 90,
            "Muito alta"
        )
        .when(
            col("percentual_participacao") >= 75,
            "Alta"
        )
        .when(
            col("percentual_participacao") >= 50,
            "Média"
        )
        .otherwise(
            "Baixa"
        )
    )
)

In [ ]:
from pyspark.sql.functions import col, when, round, dense_rank
from pyspark.sql.window import Window

janela_meta_uf = (
    Window
    .partitionBy("ano")
    .orderBy(
        col("percentual_cumprimento_meta_2024").desc_nulls_last()
    )
)

gold_meta_uf = (
    meta_uf

    .withColumn(
        "crescimento_meta_2024_2030",
        round(
            col("meta_alfabetizacao_2030")
            - col("meta_alfabetizacao_2024"),
            2
        )
    )

    .withColumn(
        "percentual_evolucao_meta",
        when(
            col("meta_alfabetizacao_2024").isNull()
            | (col("meta_alfabetizacao_2024") == 0),
            None
        ).otherwise(
            round(
                (
                    (
                        col("meta_alfabetizacao_2030")
                        - col("meta_alfabetizacao_2024")
                    )
                    / col("meta_alfabetizacao_2024")
                ) * 100,
                2
            )
        )
    )

    .withColumn(
        "diferenca_meta_2024",
        round(
            col("taxa_alfabetizacao")
            - col("meta_alfabetizacao_2024"),
            2
        )
    )

    .withColumn(
        "percentual_cumprimento_meta_2024",
        when(
            col("meta_alfabetizacao_2024").isNull()
            | (col("meta_alfabetizacao_2024") == 0),
            None
        ).otherwise(
            round(
                (
                    col("taxa_alfabetizacao")
                    / col("meta_alfabetizacao_2024")
                ) * 100,
                2
            )
        )
    )

    .withColumn(
        "status_meta_2024",
        when(
            col("meta_alfabetizacao_2024").isNull(),
            "Sem meta disponível"
        )
        .when(
            col("diferenca_meta_2024") >= 0,
            "Meta atingida"
        )
        .when(
            col("diferenca_meta_2024") >= -2,
            "Próximo da meta"
        )
        .otherwise(
            "Abaixo da meta"
        )
    )

    .withColumn(
        "faixa_participacao",
        when(
            col("percentual_participacao").isNull(),
            "Não informado"
        )
        .when(
            col("percentual_participacao") >= 90,
            "Muito alta"
        )
        .when(
            col("percentual_participacao") >= 75,
            "Alta"
        )
        .when(
            col("percentual_participacao") >= 50,
            "Média"
        )
        .otherwise(
            "Baixa"
        )
    )

    .withColumn(
        "ranking_cumprimento_meta",
        dense_rank().over(janela_meta_uf)
    )
)

In [ ]:
gold_meta_uf.select(
    "ano",
    "sigla_uf",
    "rede",
    "taxa_alfabetizacao",
    "meta_alfabetizacao_2024",
    "diferenca_meta_2024",
    "percentual_cumprimento_meta_2024",
    "status_meta_2024",
    "percentual_participacao",
    "faixa_participacao",
    "ranking_cumprimento_meta",
    "crescimento_meta_2024_2030"
).orderBy(
    "ano",
    "ranking_cumprimento_meta"
).show(30, truncate=False)

+----+--------+-------+------------------+-----------------------+-------------------+--------------------------------+-------------------+-----------------------+------------------+------------------------+--------------------------+
|ano |sigla_uf|rede   |taxa_alfabetizacao|meta_alfabetizacao_2024|diferenca_meta_2024|percentual_cumprimento_meta_2024|status_meta_2024   |percentual_participacao|faixa_participacao|ranking_cumprimento_meta|crescimento_meta_2024_2030|
+----+--------+-------+------------------+-----------------------+-------------------+--------------------------------+-------------------+-----------------------+------------------+------------------------+--------------------------+
|2023|CE      |Pública|84.5              |80.0                   |4.5                |105.62                          |Meta atingida      |95.22                  |Muito alta        |1                       |0.0                       |
|2023|PR      |Pública|73.12             |74.2              

In [ ]:
gold_meta_uf.groupBy(
    "ano",
    "status_meta_2024"
).count().orderBy(
    "ano",
    "status_meta_2024"
).show(truncate=False)

+----+-------------------+-----+
|ano |status_meta_2024   |count|
+----+-------------------+-----+
|2023|Abaixo da meta     |21   |
|2023|Meta atingida      |1    |
|2023|Próximo da meta    |2    |
|2023|Sem meta disponível|3    |
|2024|Abaixo da meta     |8    |
|2024|Meta atingida      |11   |
|2024|Próximo da meta    |5    |
|2024|Sem meta disponível|3    |
|2025|Abaixo da meta     |2    |
|2025|Meta atingida      |22   |
|2025|Sem meta disponível|3    |
+----+-------------------+-----+



In [ ]:
gold_meta_uf.select(
    round(
        avg("taxa_alfabetizacao"),
        2
    ).alias("media_taxa_alfabetizacao"),

    round(
        avg("meta_alfabetizacao_2024"),
        2
    ).alias("media_meta_2024"),

    round(
        avg("percentual_cumprimento_meta_2024"),
        2
    ).alias("media_percentual_cumprimento"),

    round(
        avg("percentual_participacao"),
        2
    ).alias("media_participacao"),

    count("*").alias("total_registros")
).show()

+------------------------+---------------+----------------------------+------------------+---------------+
|media_taxa_alfabetizacao|media_meta_2024|media_percentual_cumprimento|media_participacao|total_registros|
+------------------------+---------------+----------------------------+------------------+---------------+
|                   59.03|          58.25|                      101.23|             87.38|             81|
+------------------------+---------------+----------------------------+------------------+---------------+



In [ ]:
gold_meta_uf.select(
    [
        count(
            when(
                col(nome_coluna).isNull(),
                nome_coluna
            )
        ).alias(nome_coluna)
        for nome_coluna in gold_meta_uf.columns
    ]
).show(truncate=False)

+--------+----+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-------------+------+-------------+----------------+---+--------------------------+------------------------+-------------------+--------------------------------+----------------+------------------+------------------------+
|sigla_uf|rede|taxa_alfabetizacao|meta_alfabetizacao_2024|meta_alfabetizacao_2025|meta_alfabetizacao_2026|meta_alfabetizacao_2027|meta_alfabetizacao_2028|meta_alfabetizacao_2029|meta_alfabetizacao_2030|percentual_participacao|data_ingestao|origem|modo_ingestao|projeto_execucao|ano|crescimento_meta_2024_2030|percentual_evolucao_meta|diferenca_meta_2024|percentual_cumprimento_meta_2024|status_meta_2024|faixa_participacao|ranking_cumprimento_meta|
+--------+----+------------------+-----------------------+-----------------------+--------------------

In [ ]:
gold_meta_uf.write.mode(
    "overwrite"
).parquet(
    f"{GOLD_PATH}/meta_uf"
)

print("Data Mart meta_uf salvo com sucesso")

Data Mart meta_uf salvo com sucesso


In [ ]:
gold_meta_brasil = meta_brasil

In [ ]:
gold_meta_brasil = (
    gold_meta_brasil
    .withColumn(
        "crescimento_meta_2024_2030",
        round(
            col("meta_alfabetizacao_2030")
            - col("meta_alfabetizacao_2024"),
            2
        )
    )
)

In [ ]:
gold_meta_brasil = (
    gold_meta_brasil
    .withColumn(
        "percentual_evolucao_meta",
        when(
            col("meta_alfabetizacao_2024").isNull()
            | (col("meta_alfabetizacao_2024") == 0),
            None
        ).otherwise(
            round(
                (
                    (
                        col("meta_alfabetizacao_2030")
                        - col("meta_alfabetizacao_2024")
                    )
                    / col("meta_alfabetizacao_2024")
                ) * 100,
                2
            )
        )
    )
)

In [ ]:
gold_meta_brasil = (
    gold_meta_brasil
    .withColumn(
        "diferenca_meta_2024",
        round(
            col("taxa_alfabetizacao")
            - col("meta_alfabetizacao_2024"),
            2
        )
    )
)

In [ ]:
gold_meta_brasil = (
    gold_meta_brasil
    .withColumn(
        "percentual_cumprimento_meta_2024",
        when(
            col("meta_alfabetizacao_2024").isNull()
            | (col("meta_alfabetizacao_2024") == 0),
            None
        ).otherwise(
            round(
                (
                    col("taxa_alfabetizacao")
                    / col("meta_alfabetizacao_2024")
                ) * 100,
                2
            )
        )
    )
)

In [ ]:
gold_meta_brasil = (
    gold_meta_brasil
    .withColumn(
        "status_meta_2024",
        when(
            col("meta_alfabetizacao_2024").isNull(),
            "Sem meta disponível"
        )
        .when(
            col("diferenca_meta_2024") >= 0,
            "Meta atingida"
        )
        .when(
            col("diferenca_meta_2024") >= -2,
            "Próximo da meta"
        )
        .otherwise(
            "Abaixo da meta"
        )
    )
)

In [ ]:
gold_meta_brasil = (
    gold_meta_brasil
    .withColumn(
        "faixa_participacao",
        when(
            col("percentual_participacao").isNull(),
            "Não informado"
        )
        .when(
            col("percentual_participacao") >= 90,
            "Muito alta"
        )
        .when(
            col("percentual_participacao") >= 75,
            "Alta"
        )
        .when(
            col("percentual_participacao") >= 50,
            "Média"
        )
        .otherwise(
            "Baixa"
        )
    )
)

In [ ]:
gold_meta_brasil = (
    gold_meta_brasil
    .withColumn(
        "tendencia_resultado",
        when(
            col("diferenca_meta_2024") > 0,
            "Acima da meta"
        )
        .when(
            col("diferenca_meta_2024") == 0,
            "Na meta"
        )
        .when(
            col("diferenca_meta_2024").isNull(),
            "Sem comparação"
        )
        .otherwise(
            "Abaixo da meta"
        )
    )
)

In [ ]:
gold_meta_brasil.select(
    "ano",
    "rede",
    "taxa_alfabetizacao",
    "meta_alfabetizacao_2024",
    "meta_alfabetizacao_2025",
    "meta_alfabetizacao_2026",
    "meta_alfabetizacao_2027",
    "meta_alfabetizacao_2028",
    "meta_alfabetizacao_2029",
    "meta_alfabetizacao_2030",
    "diferenca_meta_2024",
    "percentual_cumprimento_meta_2024",
    "status_meta_2024",
    "tendencia_resultado",
    "percentual_participacao",
    "faixa_participacao",
    "crescimento_meta_2024_2030",
    "percentual_evolucao_meta"
).orderBy("ano").show(truncate=False)

+----+-------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-------------------+--------------------------------+----------------+-------------------+-----------------------+------------------+--------------------------+------------------------+
|ano |rede   |taxa_alfabetizacao|meta_alfabetizacao_2024|meta_alfabetizacao_2025|meta_alfabetizacao_2026|meta_alfabetizacao_2027|meta_alfabetizacao_2028|meta_alfabetizacao_2029|meta_alfabetizacao_2030|diferenca_meta_2024|percentual_cumprimento_meta_2024|status_meta_2024|tendencia_resultado|percentual_participacao|faixa_participacao|crescimento_meta_2024_2030|percentual_evolucao_meta|
+----+-------+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-------------------+-----

In [ ]:
gold_meta_brasil.select(
    round(
        avg("taxa_alfabetizacao"),
        2
    ).alias("media_taxa_alfabetizacao"),

    round(
        avg("meta_alfabetizacao_2024"),
        2
    ).alias("media_meta_2024"),

    round(
        avg("percentual_cumprimento_meta_2024"),
        2
    ).alias("media_percentual_cumprimento"),

    round(
        avg("percentual_participacao"),
        2
    ).alias("media_participacao"),

    round(
        avg("crescimento_meta_2024_2030"),
        2
    ).alias("crescimento_medio_meta"),

    count("*").alias("total_registros")
).show()

+------------------------+---------------+----------------------------+------------------+----------------------+---------------+
|media_taxa_alfabetizacao|media_meta_2024|media_percentual_cumprimento|media_participacao|crescimento_medio_meta|total_registros|
+------------------------+---------------+----------------------------+------------------+----------------------+---------------+
|                   60.37|          59.93|                      100.72|             87.12|                 20.07|              3|
+------------------------+---------------+----------------------------+------------------+----------------------+---------------+



In [ ]:
gold_meta_brasil.groupBy(
    "ano",
    "status_meta_2024"
).count().orderBy(
    "ano",
    "status_meta_2024"
).show(truncate=False)

+----+----------------+-----+
|ano |status_meta_2024|count|
+----+----------------+-----+
|2023|Abaixo da meta  |1    |
|2024|Próximo da meta |1    |
|2025|Meta atingida   |1    |
+----+----------------+-----+



In [ ]:
gold_meta_brasil.select(
    [
        count(
            when(
                col(nome_coluna).isNull(),
                nome_coluna
            )
        ).alias(nome_coluna)
        for nome_coluna in gold_meta_brasil.columns
    ]
).show(truncate=False)

+----+------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-----------------------+-------------+------+-------------+----------------+---+--------------------------+------------------------+-------------------+--------------------------------+----------------+------------------+-------------------+
|rede|taxa_alfabetizacao|meta_alfabetizacao_2024|meta_alfabetizacao_2025|meta_alfabetizacao_2026|meta_alfabetizacao_2027|meta_alfabetizacao_2028|meta_alfabetizacao_2029|meta_alfabetizacao_2030|percentual_participacao|data_ingestao|origem|modo_ingestao|projeto_execucao|ano|crescimento_meta_2024_2030|percentual_evolucao_meta|diferenca_meta_2024|percentual_cumprimento_meta_2024|status_meta_2024|faixa_participacao|tendencia_resultado|
+----+------------------+-----------------------+-----------------------+-----------------------+-----------------------+---------

In [ ]:
gold_meta_brasil.write.mode(
    "overwrite"
).parquet(
    f"{GOLD_PATH}/meta_brasil"
)

print("Data Mart meta_brasil salvo com sucesso")

Data Mart meta_brasil salvo com sucesso


In [ ]:
from datetime import datetime

from pyspark.sql.functions import (
    avg,
    col,
    count,
    countDistinct,
    desc,
    lit,
    max as spark_max,
    round as spark_round,
    when
)

from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType
)

In [ ]:
ano_mais_recente = (
    gold_municipio
    .select(
        spark_max("ano").alias("ano_mais_recente")
    )
    .first()["ano_mais_recente"]
)

print(f"Ano mais recente disponível: {ano_mais_recente}")

Ano mais recente disponível: 2024


In [ ]:
municipio_recente = gold_municipio.filter(
    col("ano") == ano_mais_recente
)

uf_recente = gold_uf.filter(
    col("ano") == ano_mais_recente
)

meta_uf_recente = gold_meta_uf.filter(
    col("ano") == ano_mais_recente
)

meta_brasil_recente = gold_meta_brasil.filter(
    col("ano") == ano_mais_recente
)

In [ ]:
total_municipios = (
    municipio_recente
    .select(
        countDistinct("id_municipio").alias("total")
    )
    .first()["total"]
)

total_ufs = (
    uf_recente
    .select(
        countDistinct("sigla_uf").alias("total")
    )
    .first()["total"]
)

total_registros_municipais = municipio_recente.count()

total_registros_uf = uf_recente.count()

print("Municípios distintos:", total_municipios)
print("UFs distintas:", total_ufs)
print("Registros municipais:", total_registros_municipais)
print("Registros de UF:", total_registros_uf)

Municípios distintos: 5516
UFs distintas: 25
Registros municipais: 12448
Registros de UF: 75


In [ ]:
medias_municipais = (
    municipio_recente
    .agg(
        spark_round(
            avg("taxa_alfabetizacao"),
            2
        ).alias("media_alfabetizacao"),

        spark_round(
            avg("media_portugues"),
            2
        ).alias("media_portugues"),

        spark_round(
            avg("indice_geral"),
            2
        ).alias("media_indice_geral")
    )
    .first()
)

media_alfabetizacao_municipal = medias_municipais["media_alfabetizacao"]
media_portugues_municipal = medias_municipais["media_portugues"]
media_indice_municipal = medias_municipais["media_indice_geral"]

In [ ]:
medias_uf = (
    uf_recente
    .agg(
        spark_round(
            avg("taxa_alfabetizacao"),
            2
        ).alias("media_alfabetizacao"),

        spark_round(
            avg("media_portugues"),
            2
        ).alias("media_portugues"),

        spark_round(
            avg("indice_geral"),
            2
        ).alias("media_indice_geral")
    )
    .first()
)

media_alfabetizacao_uf = medias_uf["media_alfabetizacao"]
media_portugues_uf = medias_uf["media_portugues"]
media_indice_uf = medias_uf["media_indice_geral"]

In [ ]:
distribuicao_desempenho = (
    municipio_recente
    .groupBy("nivel_desempenho")
    .agg(
        count("*").alias("quantidade")
    )
    .withColumn(
        "percentual",
        spark_round(
            col("quantidade")
            / lit(total_registros_municipais)
            * 100,
            2
        )
    )
)

distribuicao_desempenho.show(truncate=False)

+----------------+----------+----------+
|nivel_desempenho|quantidade|percentual|
+----------------+----------+----------+
|Crítico         |5545      |44.55     |
|Bom             |2170      |17.43     |
|Excelente       |2540      |20.4      |
|Regular         |2193      |17.62     |
+----------------+----------+----------+



In [ ]:
percentuais_desempenho = {
    linha["nivel_desempenho"]: linha["percentual"]
    for linha in distribuicao_desempenho.collect()
}

percentual_excelente = percentuais_desempenho.get("Excelente", 0.0)
percentual_bom = percentuais_desempenho.get("Bom", 0.0)
percentual_regular = percentuais_desempenho.get("Regular", 0.0)
percentual_critico = percentuais_desempenho.get("Crítico", 0.0)

In [ ]:
melhor_municipio = (
    municipio_recente
    .filter(
        col("indice_geral").isNotNull()
    )
    .orderBy(
        desc("indice_geral")
    )
    .select(
        "id_municipio",
        "categoria_rede",
        "indice_geral",
        "taxa_alfabetizacao",
        "media_portugues"
    )
    .first()
)

print(melhor_municipio)

Row(id_municipio='2304004', categoria_rede='Privada', indice_geral=93.4, taxa_alfabetizacao=100.0, media_portugues=868.01)


In [ ]:
melhor_uf = (
    uf_recente
    .filter(
        col("indice_geral").isNotNull()
    )
    .orderBy(
        desc("indice_geral")
    )
    .select(
        "sigla_uf",
        "categoria_rede",
        "indice_geral",
        "taxa_alfabetizacao",
        "media_portugues"
    )
    .first()
)

print(melhor_uf)

Row(sigla_uf='CE', categoria_rede='Municipal', indice_geral=82.54, taxa_alfabetizacao=85.35, media_portugues=797.34)


In [ ]:
indicadores_meta_uf = (
    meta_uf_recente
    .agg(
        spark_round(
            avg("percentual_cumprimento_meta_2024"),
            2
        ).alias("media_cumprimento"),

        spark_round(
            avg("percentual_participacao"),
            2
        ).alias("media_participacao"),

        count(
            when(
                col("status_meta_2024") == "Meta atingida",
                True
            )
        ).alias("ufs_meta_atingida")
    )
    .first()
)

media_cumprimento_meta_uf = indicadores_meta_uf["media_cumprimento"]
media_participacao_uf = indicadores_meta_uf["media_participacao"]
ufs_meta_atingida = indicadores_meta_uf["ufs_meta_atingida"]

In [ ]:
indicador_nacional = (
    meta_brasil_recente
    .select(
        "taxa_alfabetizacao",
        "meta_alfabetizacao_2024",
        "percentual_cumprimento_meta_2024",
        "status_meta_2024",
        "percentual_participacao"
    )
    .first()
)

print(indicador_nacional)

Row(taxa_alfabetizacao=59.2, meta_alfabetizacao_2024=59.9, percentual_cumprimento_meta_2024=98.83, status_meta_2024='Próximo da meta', percentual_participacao=87.37)


In [ ]:
from pyspark.sql.functions import (
    avg,
    col,
    count,
    countDistinct,
    lit,
    max as spark_max,
    min as spark_min,
    round as spark_round,
    sum as spark_sum,
    when
)

In [ ]:
gold_perfil_alunos = alunos_agregados

In [ ]:
gold_perfil_alunos = (
    gold_perfil_alunos
    .withColumn(
        "nivel_presenca",
        when(
            col("taxa_presenca").isNull(),
            "Não informado"
        )
        .when(
            col("taxa_presenca") >= 90,
            "Muito alta"
        )
        .when(
            col("taxa_presenca") >= 75,
            "Alta"
        )
        .when(
            col("taxa_presenca") >= 50,
            "Média"
        )
        .otherwise(
            "Baixa"
        )
    )
)

In [ ]:
gold_perfil_alunos = (
    gold_perfil_alunos
    .withColumn(
        "nivel_alfabetizacao_alunos",
        when(
            col("taxa_alfabetizacao_alunos").isNull(),
            "Não informado"
        )
        .when(
            col("taxa_alfabetizacao_alunos") >= 80,
            "Excelente"
        )
        .when(
            col("taxa_alfabetizacao_alunos") >= 70,
            "Bom"
        )
        .when(
            col("taxa_alfabetizacao_alunos") >= 60,
            "Regular"
        )
        .otherwise(
            "Crítico"
        )
    )
)

In [ ]:
PONTO_CORTE_ALFABETIZACAO = 743.0

In [ ]:
gold_perfil_alunos = (
    gold_perfil_alunos
    .withColumn(
        "situacao_proficiencia",
        when(
            col("proficiencia_media").isNull(),
            "Não informado"
        )
        .when(
            col("proficiencia_media")
            >= lit(PONTO_CORTE_ALFABETIZACAO),
            "Acima do ponto de corte"
        )
        .otherwise(
            "Abaixo do ponto de corte"
        )
    )
    .withColumn(
        "diferenca_ponto_corte",
        spark_round(
            col("proficiencia_media")
            - lit(PONTO_CORTE_ALFABETIZACAO),
            2
        )
    )
)

In [ ]:
gold_perfil_alunos = (
    gold_perfil_alunos
    .withColumn(
        "indice_participacao_aprendizagem",
        spark_round(
            (
                col("taxa_presenca")
                + col("taxa_alfabetizacao_alunos")
                + col("taxa_preenchimento_caderno")
            ) / 3,
            2
        )
    )
)

In [ ]:
gold_perfil_alunos = (
    gold_perfil_alunos
    .withColumn(
        "classificacao_indice",
        when(
            col("indice_participacao_aprendizagem").isNull(),
            "Não informado"
        )
        .when(
            col("indice_participacao_aprendizagem") >= 80,
            "Muito alto"
        )
        .when(
            col("indice_participacao_aprendizagem") >= 70,
            "Alto"
        )
        .when(
            col("indice_participacao_aprendizagem") >= 60,
            "Médio"
        )
        .otherwise(
            "Baixo"
        )
    )
)

In [ ]:
gold_perfil_alunos.select(
    "ano",
    "id_municipio",
    "id_escola",
    "serie",
    "rede",
    "total_registros",
    "total_presentes",
    "total_alfabetizados",
    "taxa_presenca",
    "taxa_alfabetizacao_alunos",
    "taxa_preenchimento_caderno",
    "proficiencia_media",
    "situacao_proficiencia",
    "diferenca_ponto_corte",
    "indice_participacao_aprendizagem",
    "classificacao_indice"
).show(
    20,
    truncate=False
)

+----+------------+---------+-----+----+---------------+---------------+-------------------+-----------------+-------------------------+--------------------------+------------------+------------------------+---------------------+--------------------------------+--------------------+
|ano |id_municipio|id_escola|serie|rede|total_registros|total_presentes|total_alfabetizados|taxa_presenca    |taxa_alfabetizacao_alunos|taxa_preenchimento_caderno|proficiencia_media|situacao_proficiencia   |diferenca_ponto_corte|indice_participacao_aprendizagem|classificacao_indice|
+----+------------+---------+-----+----+---------------+---------------+-------------------+-----------------+-------------------------+--------------------------+------------------+------------------------+---------------------+--------------------------------+--------------------+
|2024|3550308     |60025856 |2    |2   |123            |108            |60                 |87.8048780487805 |48.78048780487805        |87.804878048

In [ ]:
gold_perfil_alunos.select(
    spark_sum(
        "total_registros"
    ).alias(
        "total_registros_alunos"
    ),

    spark_sum(
        "total_presentes"
    ).alias(
        "total_presentes"
    ),

    spark_sum(
        "total_alfabetizados"
    ).alias(
        "total_alfabetizados"
    ),

    spark_round(
        avg("taxa_presenca"),
        2
    ).alias(
        "media_taxa_presenca"
    ),

    spark_round(
        avg("taxa_alfabetizacao_alunos"),
        2
    ).alias(
        "media_taxa_alfabetizacao"
    ),

    spark_round(
        avg("proficiencia_media"),
        2
    ).alias(
        "media_proficiencia"
    ),

    countDistinct(
        "id_escola"
    ).alias(
        "escolas_distintas"
    ),

    countDistinct(
        "id_municipio"
    ).alias(
        "municipios_distintos"
    )
).show(
    truncate=False
)

+----------------------+---------------+-------------------+-------------------+------------------------+------------------+-----------------+--------------------+
|total_registros_alunos|total_presentes|total_alfabetizados|media_taxa_presenca|media_taxa_alfabetizacao|media_proficiencia|escolas_distintas|municipios_distintos|
+----------------------+---------------+-------------------+-------------------+------------------------+------------------+-----------------+--------------------+
|3867999               |3355846        |1984546            |87.47              |51.69                   |NaN               |42811            |5548                |
+----------------------+---------------+-------------------+-------------------+------------------------+------------------+-----------------+--------------------+



In [ ]:
gold_perfil_alunos.groupBy(
    "ano",
    "situacao_proficiencia"
).agg(
    count("*").alias(
        "grupos_avaliados"
    ),
    spark_sum(
        "total_registros"
    ).alias(
        "alunos_representados"
    )
).orderBy(
    "ano",
    "situacao_proficiencia"
).show(
    truncate=False
)

+----+------------------------+----------------+--------------------+
|ano |situacao_proficiencia   |grupos_avaliados|alunos_representados|
+----+------------------------+----------------+--------------------+
|2023|Abaixo do ponto de corte|16581           |799629              |
|2023|Acima do ponto de corte |20195           |947810              |
|2024|Abaixo do ponto de corte|18123           |897396              |
|2024|Acima do ponto de corte |24374           |1223164             |
+----+------------------------+----------------+--------------------+



In [ ]:
gold_perfil_alunos.groupBy(
    "ano",
    "nivel_alfabetizacao_alunos"
).agg(
    count("*").alias(
        "grupos_avaliados"
    ),
    spark_sum(
        "total_registros"
    ).alias(
        "alunos_representados"
    )
).orderBy(
    "ano",
    "nivel_alfabetizacao_alunos"
).show(
    truncate=False
)

+----+--------------------------+----------------+--------------------+
|ano |nivel_alfabetizacao_alunos|grupos_avaliados|alunos_representados|
+----+--------------------------+----------------+--------------------+
|2023|Bom                       |3562            |161200              |
|2023|Crítico                   |23677           |1183289             |
|2023|Excelente                 |4590            |154727              |
|2023|Regular                   |4947            |248223              |
|2024|Bom                       |4458            |219281              |
|2024|Crítico                   |26479           |1374629             |
|2024|Excelente                 |5490            |204805              |
|2024|Regular                   |6070            |321845              |
+----+--------------------------+----------------+--------------------+



In [ ]:
COLUNAS_PROIBIDAS = {
    "id_aluno",
    "id_aluno_hash"
}

colunas_encontradas = (
    COLUNAS_PROIBIDAS
    .intersection(
        set(gold_perfil_alunos.columns)
    )
)

assert not colunas_encontradas, (
    "Foram encontradas colunas individuais na Gold: "
    f"{colunas_encontradas}"
)

print(
    "Gold de alunos contém apenas dados agregados."
)

Gold de alunos contém apenas dados agregados.


In [ ]:
(
    gold_perfil_alunos
    .write
    .mode("overwrite")
    .partitionBy("ano")
    .parquet(
        f"{GOLD_PATH}/perfil_alunos"
    )
)

print(
    "Data Mart perfil_alunos salvo com sucesso!"
)

Data Mart perfil_alunos salvo com sucesso!


In [ ]:
perfil_alunos_validacao = spark.read.parquet(
    f"{GOLD_PATH}/perfil_alunos"
)

print(
    "Registros:",
    perfil_alunos_validacao.count()
)

print(
    "Colunas:",
    len(perfil_alunos_validacao.columns)
)

perfil_alunos_validacao.printSchema()

Registros: 79273
Colunas: 24
root
 |-- id_municipio: string (nullable = true)
 |-- id_escola: string (nullable = true)
 |-- serie: string (nullable = true)
 |-- rede: string (nullable = true)
 |-- total_registros: long (nullable = true)
 |-- total_presentes: long (nullable = true)
 |-- total_alfabetizados: long (nullable = true)
 |-- total_cadernos_preenchidos: long (nullable = true)
 |-- proficiencia_media: double (nullable = true)
 |-- peso_medio: double (nullable = true)
 |-- taxa_presenca: double (nullable = true)
 |-- taxa_alfabetizacao_alunos: double (nullable = true)
 |-- data_ingestao: timestamp (nullable = true)
 |-- origem: string (nullable = true)
 |-- modo_ingestao: string (nullable = true)
 |-- granularidade: string (nullable = true)
 |-- taxa_preenchimento_caderno: double (nullable = true)
 |-- nivel_presenca: string (nullable = true)
 |-- nivel_alfabetizacao_alunos: string (nullable = true)
 |-- situacao_proficiencia: string (nullable = true)
 |-- diferenca_ponto_corte: 

In [ ]:
from datetime import datetime
import builtins

from pyspark.sql.functions import (
    avg,
    col,
    count,
    countDistinct,
    desc,
    lit,
    max as spark_max,
    round as spark_round,
    sum as spark_sum,
    when
)

from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
    TimestampType
)

In [ ]:
gold_municipio = spark.read.parquet(
    f"{GOLD_PATH}/desempenho_municipio"
)

gold_uf = spark.read.parquet(
    f"{GOLD_PATH}/desempenho_uf"
)

gold_meta_uf = spark.read.parquet(
    f"{GOLD_PATH}/meta_uf"
)

gold_meta_brasil = spark.read.parquet(
    f"{GOLD_PATH}/meta_brasil"
)

gold_perfil_alunos = spark.read.parquet(
    f"{GOLD_PATH}/perfil_alunos"
)

print("data Marts carregados.")

data Marts carregados.


In [ ]:
COLUNAS_ESPERADAS = {
    "desempenho_municipio": {
        "ano",
        "id_municipio",
        "taxa_alfabetizacao",
        "media_portugues",
        "indice_geral",
        "nivel_desempenho"
    },
    "desempenho_uf": {
        "ano",
        "sigla_uf",
        "taxa_alfabetizacao",
        "media_portugues",
        "indice_geral"
    },
    "meta_uf": {
        "ano",
        "sigla_uf",
        "percentual_cumprimento_meta_2024",
        "status_meta_2024",
        "percentual_participacao"
    },
    "meta_brasil": {
        "ano",
        "taxa_alfabetizacao",
        "meta_alfabetizacao_2024",
        "percentual_cumprimento_meta_2024",
        "status_meta_2024"
    },
    "perfil_alunos": {
        "ano",
        "id_municipio",
        "id_escola",
        "total_registros",
        "total_presentes",
        "total_alfabetizados",
        "total_cadernos_preenchidos",
        "taxa_presenca",
        "taxa_alfabetizacao_alunos",
        "taxa_preenchimento_caderno",
        "proficiencia_media",
        "situacao_proficiencia"
    }
}

DATA_MARTS = {
    "desempenho_municipio": gold_municipio,
    "desempenho_uf": gold_uf,
    "meta_uf": gold_meta_uf,
    "meta_brasil": gold_meta_brasil,
    "perfil_alunos": gold_perfil_alunos
}

for nome, df in DATA_MARTS.items():
    faltantes = (
        COLUNAS_ESPERADAS[nome]
        - set(df.columns)
    )

    if faltantes:
        raise ValueError(
            f"{nome}: colunas ausentes: {sorted(faltantes)}"
        )

    print(f"{nome}: schema validado.")

desempenho_municipio: schema validado.
desempenho_uf: schema validado.
meta_uf: schema validado.
meta_brasil: schema validado.
perfil_alunos: schema validado.


In [ ]:
anos_recentes = []

for nome, df in DATA_MARTS.items():
    ano_maximo = (
        df.select(
            spark_max("ano").alias("ano_maximo")
        )
        .first()["ano_maximo"]
    )

    anos_recentes.append(
        {
            "data_mart": nome,
            "ano_maximo": ano_maximo
        }
    )

anos_recentes

[{'data_mart': 'desempenho_municipio', 'ano_maximo': 2024},
 {'data_mart': 'desempenho_uf', 'ano_maximo': 2024},
 {'data_mart': 'meta_uf', 'ano_maximo': 2025},
 {'data_mart': 'meta_brasil', 'ano_maximo': 2025},
 {'data_mart': 'perfil_alunos', 'ano_maximo': 2024}]

In [ ]:
ANO_REFERENCIA = (
    gold_municipio
    .select(
        spark_max("ano").alias("ano")
    )
    .first()["ano"]
)

print(f"Ano de referência dos KPIs: {ANO_REFERENCIA}")

Ano de referência dos KPIs: 2024


In [ ]:
municipio_recente = gold_municipio.filter(
    col("ano") == ANO_REFERENCIA
)

uf_recente = gold_uf.filter(
    col("ano") == ANO_REFERENCIA
)

meta_uf_recente = gold_meta_uf.filter(
    col("ano") == ANO_REFERENCIA
)

meta_brasil_recente = gold_meta_brasil.filter(
    col("ano") == ANO_REFERENCIA
)

alunos_recente = gold_perfil_alunos.filter(
    col("ano") == ANO_REFERENCIA
)

In [ ]:
for nome, df in {
    "municipio": municipio_recente,
    "uf": uf_recente,
    "meta_uf": meta_uf_recente,
    "meta_brasil": meta_brasil_recente,
    "alunos": alunos_recente
}.items():
    print(f"{nome}: {df.count():,} registros")

municipio: 12,448 registros
uf: 75 registros
meta_uf: 27 registros
meta_brasil: 1 registros
alunos: 42,497 registros


In [ ]:
total_municipios = (
    municipio_recente
    .select(
        countDistinct("id_municipio").alias("total")
    )
    .first()["total"]
)

total_ufs = (
    uf_recente
    .select(
        countDistinct("sigla_uf").alias("total")
    )
    .first()["total"]
)

total_escolas = (
    alunos_recente
    .select(
        countDistinct("id_escola").alias("total")
    )
    .first()["total"]
)

total_municipios_alunos = (
    alunos_recente
    .select(
        countDistinct("id_municipio").alias("total")
    )
    .first()["total"]
)

print("Municípios:", total_municipios)
print("UFs:", total_ufs)
print("Escolas representadas:", total_escolas)
print("Municípios na base de alunos:", total_municipios_alunos)

Municípios: 5516
UFs: 25
Escolas representadas: 42497
Municípios na base de alunos: 5519


In [ ]:
medias_municipais = (
    municipio_recente
    .agg(
        spark_round(
            avg("taxa_alfabetizacao"),
            2
        ).alias("media_alfabetizacao"),

        spark_round(
            avg("media_portugues"),
            2
        ).alias("media_portugues"),

        spark_round(
            avg("indice_geral"),
            2
        ).alias("media_indice_geral"),

        count("*").alias("total_registros")
    )
    .first()
)

media_alfabetizacao_municipal = (
    medias_municipais["media_alfabetizacao"]
)

media_portugues_municipal = (
    medias_municipais["media_portugues"]
)

media_indice_municipal = (
    medias_municipais["media_indice_geral"]
)

total_registros_municipais = (
    medias_municipais["total_registros"]
)

In [ ]:
distribuicao_desempenho = (
    municipio_recente
    .groupBy("nivel_desempenho")
    .agg(
        count("*").alias("quantidade")
    )
    .withColumn(
        "percentual",
        spark_round(
            (
                col("quantidade")
                / lit(total_registros_municipais)
            ) * 100,
            2
        )
    )
)

distribuicao_desempenho.show(
    truncate=False
)

+----------------+----------+----------+
|nivel_desempenho|quantidade|percentual|
+----------------+----------+----------+
|Crítico         |5545      |44.55     |
|Bom             |2170      |17.43     |
|Excelente       |2540      |20.4      |
|Regular         |2193      |17.62     |
+----------------+----------+----------+



In [ ]:
percentuais_desempenho = {
    linha["nivel_desempenho"]: linha["percentual"]
    for linha in distribuicao_desempenho.collect()
}

percentual_excelente = (
    percentuais_desempenho.get("Excelente", 0.0)
)

percentual_bom = (
    percentuais_desempenho.get("Bom", 0.0)
)

percentual_regular = (
    percentuais_desempenho.get("Regular", 0.0)
)

percentual_critico = (
    percentuais_desempenho.get("Crítico", 0.0)
)

In [ ]:
melhor_municipio = (
    municipio_recente
    .filter(
        col("indice_geral").isNotNull()
    )
    .orderBy(
        desc("indice_geral")
    )
    .select(
        "id_municipio",
        "categoria_rede",
        "indice_geral",
        "taxa_alfabetizacao",
        "media_portugues"
    )
    .first()
)

print(melhor_municipio)

Row(id_municipio='2304004', categoria_rede='Privada', indice_geral=93.4, taxa_alfabetizacao=100.0, media_portugues=868.01)


In [ ]:
medias_uf = (
    uf_recente
    .agg(
        spark_round(
            avg("taxa_alfabetizacao"),
            2
        ).alias("media_alfabetizacao"),

        spark_round(
            avg("media_portugues"),
            2
        ).alias("media_portugues"),

        spark_round(
            avg("indice_geral"),
            2
        ).alias("media_indice_geral")
    )
    .first()
)

media_alfabetizacao_uf = (
    medias_uf["media_alfabetizacao"]
)

media_portugues_uf = (
    medias_uf["media_portugues"]
)

media_indice_uf = (
    medias_uf["media_indice_geral"]
)

In [ ]:
melhor_uf = (
    uf_recente
    .filter(
        col("indice_geral").isNotNull()
    )
    .orderBy(
        desc("indice_geral")
    )
    .select(
        "sigla_uf",
        "categoria_rede",
        "indice_geral",
        "taxa_alfabetizacao",
        "media_portugues"
    )
    .first()
)

print(melhor_uf)

Row(sigla_uf='CE', categoria_rede='Municipal', indice_geral=82.54, taxa_alfabetizacao=85.35, media_portugues=797.34)


In [ ]:
indicadores_meta_uf = (
    meta_uf_recente
    .agg(
        spark_round(
            avg("percentual_cumprimento_meta_2024"),
            2
        ).alias("media_cumprimento"),

        spark_round(
            avg("percentual_participacao"),
            2
        ).alias("media_participacao"),

        count(
            when(
                col("status_meta_2024")
                == "Meta atingida",
                True
            )
        ).alias("ufs_meta_atingida")
    )
    .first()
)

media_cumprimento_meta_uf = (
    indicadores_meta_uf["media_cumprimento"]
)

media_participacao_uf = (
    indicadores_meta_uf["media_participacao"]
)

ufs_meta_atingida = (
    indicadores_meta_uf["ufs_meta_atingida"]
)

In [ ]:
print(
    "Registros nacionais no ano:",
    meta_brasil_recente.count()
)

Registros nacionais no ano: 1


In [ ]:
indicador_nacional = (
    meta_brasil_recente
    .agg(
        spark_round(
            avg("taxa_alfabetizacao"),
            2
        ).alias("taxa_alfabetizacao"),

        spark_round(
            avg("meta_alfabetizacao_2024"),
            2
        ).alias("meta_2024"),

        spark_round(
            avg("percentual_cumprimento_meta_2024"),
            2
        ).alias("percentual_cumprimento")
    )
    .first()
)

taxa_nacional = (
    indicador_nacional["taxa_alfabetizacao"]
)

meta_nacional_2024 = (
    indicador_nacional["meta_2024"]
)

cumprimento_nacional = (
    indicador_nacional["percentual_cumprimento"]
)

In [ ]:
totais_alunos = (
    alunos_recente
    .agg(
        spark_sum(
            "total_registros"
        ).alias("total_registros"),

        spark_sum(
            "total_presentes"
        ).alias("total_presentes"),

        spark_sum(
            "total_alfabetizados"
        ).alias("total_alfabetizados"),

        spark_sum(
            "total_cadernos_preenchidos"
        ).alias("total_cadernos_preenchidos")
    )
    .first()
)

total_alunos_representados = (
    totais_alunos["total_registros"]
)

total_presentes = (
    totais_alunos["total_presentes"]
)

total_alfabetizados = (
    totais_alunos["total_alfabetizados"]
)

total_cadernos_preenchidos = (
    totais_alunos["total_cadernos_preenchidos"]
)

In [ ]:
taxa_presenca_ponderada = (
    builtins.round(
        (
            total_presentes
            / total_alunos_representados
        ) * 100,
        2
    )
    if total_alunos_representados
    else None
)

taxa_alfabetizacao_ponderada = (
    builtins.round(
        (
            total_alfabetizados
            / total_alunos_representados
        ) * 100,
        2
    )
    if total_alunos_representados
    else None
)

taxa_preenchimento_ponderada = (
    builtins.round(
        (
            total_cadernos_preenchidos
            / total_alunos_representados
        ) * 100,
        2
    )
    if total_alunos_representados
    else None
)

In [ ]:
proficiencia_ponderada = (
    alunos_recente
    .filter(
        col("proficiencia_media").isNotNull()
        & col("total_registros").isNotNull()
        & (col("total_registros") > 0)
    )
    .agg(
        spark_round(
            (
                spark_sum(
                    col("proficiencia_media")
                    * col("total_registros")
                )
                / spark_sum("total_registros")
            ),
            2
        ).alias("proficiencia_media_ponderada")
    )
    .first()["proficiencia_media_ponderada"]
)

print(
    "Proficiência média ponderada:",
    proficiencia_ponderada
)

Proficiência média ponderada: nan


In [ ]:
total_grupos_alunos = alunos_recente.count()

grupos_acima_corte = (
    alunos_recente
    .filter(
        col("situacao_proficiencia")
        == "Acima do ponto de corte"
    )
    .count()
)

percentual_grupos_acima_corte = (
    builtins.round(
        (
            grupos_acima_corte
            / total_grupos_alunos
        ) * 100,
        2
    )
    if total_grupos_alunos
    else None
)

print(
    "Total de grupos:",
    total_grupos_alunos
)

print(
    "Grupos acima do ponto de corte:",
    grupos_acima_corte
)

print(
    "Percentual de grupos acima de 743:",
    percentual_grupos_acima_corte
)

Total de grupos: 42497
Grupos acima do ponto de corte: 24374
Percentual de grupos acima de 743: 57.35


In [ ]:
schema_kpis = StructType([
    StructField(
        "grupo",
        StringType(),
        False
    ),
    StructField(
        "indicador",
        StringType(),
        False
    ),
    StructField(
        "valor_numerico",
        DoubleType(),
        True
    ),
    StructField(
        "valor_texto",
        StringType(),
        True
    ),
    StructField(
        "unidade",
        StringType(),
        True
    ),
    StructField(
        "ano_referencia",
        IntegerType(),
        False
    ),
    StructField(
        "data_processamento",
        TimestampType(),
        False
    )
])

In [ ]:
def para_float(valor):
    """
    Converte valores numéricos para float,
    preservando None.
    """

    if valor is None:
        return None

    return float(valor)

In [ ]:
data_processamento = datetime.now()

kpis = [
    # Cobertura
    (
        "Cobertura",
        "Total de municípios distintos",
        para_float(total_municipios),
        None,
        "municípios",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Cobertura",
        "Total de UFs distintas",
        para_float(total_ufs),
        None,
        "UFs",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Cobertura",
        "Total de escolas representadas",
        para_float(total_escolas),
        None,
        "escolas",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Cobertura",
        "Municípios presentes na base de alunos",
        para_float(total_municipios_alunos),
        None,
        "municípios",
        int(ANO_REFERENCIA),
        data_processamento
    ),

    # Desempenho municipal
    (
        "Desempenho municipal",
        "Média da taxa de alfabetização",
        para_float(media_alfabetizacao_municipal),
        None,
        "percentual",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Desempenho municipal",
        "Média de Português",
        para_float(media_portugues_municipal),
        None,
        "pontos",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Desempenho municipal",
        "Média do índice geral",
        para_float(media_indice_municipal),
        None,
        "índice",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Desempenho municipal",
        "Registros com desempenho excelente",
        para_float(percentual_excelente),
        None,
        "percentual",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Desempenho municipal",
        "Registros com desempenho bom",
        para_float(percentual_bom),
        None,
        "percentual",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Desempenho municipal",
        "Registros com desempenho regular",
        para_float(percentual_regular),
        None,
        "percentual",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Desempenho municipal",
        "Registros com desempenho crítico",
        para_float(percentual_critico),
        None,
        "percentual",
        int(ANO_REFERENCIA),
        data_processamento
    ),

    # Desempenho UF
    (
        "Desempenho UF",
        "Média da taxa de alfabetização",
        para_float(media_alfabetizacao_uf),
        None,
        "percentual",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Desempenho UF",
        "Média de Português",
        para_float(media_portugues_uf),
        None,
        "pontos",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Desempenho UF",
        "Média do índice geral",
        para_float(media_indice_uf),
        None,
        "índice",
        int(ANO_REFERENCIA),
        data_processamento
    ),

    # Metas
    (
        "Metas estaduais",
        "Média de cumprimento da meta",
        para_float(media_cumprimento_meta_uf),
        None,
        "percentual",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Metas estaduais",
        "UFs que atingiram a meta",
        para_float(ufs_meta_atingida),
        None,
        "UFs",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Metas estaduais",
        "Média de participação",
        para_float(media_participacao_uf),
        None,
        "percentual",
        int(ANO_REFERENCIA),
        data_processamento
    ),

    # Brasil
    (
        "Brasil",
        "Taxa nacional de alfabetização",
        para_float(taxa_nacional),
        None,
        "percentual",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Brasil",
        "Meta nacional de 2024",
        para_float(meta_nacional_2024),
        None,
        "percentual",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Brasil",
        "Cumprimento da meta nacional",
        para_float(cumprimento_nacional),
        None,
        "percentual",
        int(ANO_REFERENCIA),
        data_processamento
    ),

    # Perfil de alunos
    (
        "Perfil de alunos",
        "Total de registros de alunos representados",
        para_float(total_alunos_representados),
        None,
        "registros",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Perfil de alunos",
        "Total de alunos presentes",
        para_float(total_presentes),
        None,
        "registros",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Perfil de alunos",
        "Total de alunos alfabetizados",
        para_float(total_alfabetizados),
        None,
        "registros",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Perfil de alunos",
        "Taxa ponderada de presença",
        para_float(taxa_presenca_ponderada),
        None,
        "percentual",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Perfil de alunos",
        "Taxa ponderada de alfabetização",
        para_float(taxa_alfabetizacao_ponderada),
        None,
        "percentual",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Perfil de alunos",
        "Taxa ponderada de preenchimento",
        para_float(taxa_preenchimento_ponderada),
        None,
        "percentual",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Perfil de alunos",
        "Proficiência média ponderada",
        para_float(proficiencia_ponderada),
        None,
        "pontos",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Perfil de alunos",
        "Grupos acima do ponto de corte de 743",
        para_float(percentual_grupos_acima_corte),
        None,
        "percentual",
        int(ANO_REFERENCIA),
        data_processamento
    ),

    # Rankings
    (
        "Ranking",
        "Melhor registro municipal",
        para_float(
            melhor_municipio["indice_geral"]
        ),
        (
            f'{melhor_municipio["id_municipio"]}'
            f' — {melhor_municipio["categoria_rede"]}'
        ),
        "índice",
        int(ANO_REFERENCIA),
        data_processamento
    ),
    (
        "Ranking",
        "Melhor registro de UF",
        para_float(
            melhor_uf["indice_geral"]
        ),
        (
            f'{melhor_uf["sigla_uf"]}'
            f' — {melhor_uf["categoria_rede"]}'
        ),
        "índice",
        int(ANO_REFERENCIA),
        data_processamento
    )
]

In [ ]:
gold_kpis = spark.createDataFrame(
    kpis,
    schema=schema_kpis
)

print(
    f"Data Mart criado com "
    f"{gold_kpis.count()} KPIs."
)

Data Mart criado com 30 KPIs.


In [ ]:
gold_kpis.orderBy(
    "grupo",
    "indicador"
).show(
    100,
    truncate=False
)

+--------------------+------------------------------------------+--------------+-----------------+----------+--------------+--------------------------+
|grupo               |indicador                                 |valor_numerico|valor_texto      |unidade   |ano_referencia|data_processamento        |
+--------------------+------------------------------------------+--------------+-----------------+----------+--------------+--------------------------+
|Brasil              |Cumprimento da meta nacional              |98.83         |NULL             |percentual|2024          |2026-07-14 05:52:25.536397|
|Brasil              |Meta nacional de 2024                     |59.9          |NULL             |percentual|2024          |2026-07-14 05:52:25.536397|
|Brasil              |Taxa nacional de alfabetização            |59.2          |NULL             |percentual|2024          |2026-07-14 05:52:25.536397|
|Cobertura           |Municípios presentes na base de alunos    |5519.0        |NULL    

In [ ]:
duplicados_kpis = (
    gold_kpis
    .groupBy(
        "grupo",
        "indicador",
        "ano_referencia"
    )
    .count()
    .filter(
        col("count") > 1
    )
    .count()
)

if duplicados_kpis > 0:
    raise ValueError(
        f"Foram encontrados "
        f"{duplicados_kpis} KPIs duplicados."
    )

print("Não existem KPIs duplicados.")

Não existem KPIs duplicados.


In [ ]:
gold_kpis.select(
    count(
        when(
            col("valor_numerico").isNull(),
            True
        )
    ).alias(
        "valores_numericos_nulos"
    ),
    count(
        when(
            col("grupo").isNull(),
            True
        )
    ).alias(
        "grupos_nulos"
    ),
    count(
        when(
            col("indicador").isNull(),
            True
        )
    ).alias(
        "indicadores_nulos"
    )
).show()

+-----------------------+------------+-----------------+
|valores_numericos_nulos|grupos_nulos|indicadores_nulos|
+-----------------------+------------+-----------------+
|                      0|           0|                0|
+-----------------------+------------+-----------------+



In [ ]:
conteudo_colunas = " ".join(
    gold_kpis.columns
).lower()

assert "id_aluno" not in conteudo_colunas

print(
    "Data Mart de KPIs não contém "
    "identificadores individuais."
)

Data Mart de KPIs não contém identificadores individuais.


In [ ]:
(
    gold_kpis
    .write
    .mode("overwrite")
    .partitionBy("ano_referencia")
    .parquet(
        f"{GOLD_PATH}/kpis"
    )
)

print(
    "Data Mart kpis atualizado e salvo."
)

Data Mart kpis atualizado e salvo.


In [ ]:
OUTPUT_PATH = (
    f"{BASE_PATH}/outputs"
)

(
    gold_kpis
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(
        f"{OUTPUT_PATH}/kpis_executivos"
    )
)

print(
    "Versão CSV exportada."
)

Versão CSV exportada.


In [ ]:
kpis_validacao = spark.read.parquet(
    f"{GOLD_PATH}/kpis"
)

print(
    "Registros salvos:",
    kpis_validacao.count()
)

kpis_validacao.groupBy(
    "grupo"
).count().orderBy(
    "grupo"
).show(
    truncate=False
)

Registros salvos: 30
+--------------------+-----+
|grupo               |count|
+--------------------+-----+
|Brasil              |3    |
|Cobertura           |4    |
|Desempenho UF       |3    |
|Desempenho municipal|7    |
|Metas estaduais     |3    |
|Perfil de alunos    |8    |
|Ranking             |2    |
+--------------------+-----+

